# Analyse Mt. Kenya data
Follow this procedure to analyse the level2 data

### CAMS data
The first part about CAMS data needs to be run only once and downloads the CAMS data to your local computer (taking a long time). 

It requires to have a login at the [Atmosphere data store](https://ads.atmosphere.copernicus.eu/cdsapp#!/home) and an installation of [cdsapi](https://cds.climate.copernicus.eu/api-how-to). 

This data is then read in and saved as netcdfs in the `./data/cams/` folder. 

If this was already done, this first part can be skipped. 

In [1]:
import numpy as np
import os,sys
import xarray as xr
import pandas as pd


In [10]:
## general settings
stat = 'MKN'

dir_data_cams = r"..\..\..\Data\CAMS" #adapt path to local folder to save CAMS data

In [ ]:
import get_cams #require to install cdsapi (https://cds.climate.copernicus.eu/api-how-to)

## Get CAMS data
# This may take several days, and it is only required to be done once
# It was done in Jan 2024, and the data will be saved in seperate netcdf files (see next step)

get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2003, yr2=2020,which='cams_egg4',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2003, yr2=2022,which='cams_eac4',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2020, yr2=2023,which='cams_inv_co2',station=[stat])
get_cams.main(dir_data=dir_data_cams + r"\CAMS", yr1=2020, yr2=2021,which='cams_inv_ch4',station=[stat])

In [ ]:
import analyses_level2.read_cams as read_cams

# Read in the CAMS data and save as netcdfs (one for each cams-dataset)
# The netcdfs will be saved in .\data\cams\

#those take about a minute:
read_cams.read_cams_inv(dir_data_cams,species='co2',yr1=2020,yr2=2023,station=stat)
read_cams.read_cams_inv(dir_data_cams,species='ch4',yr1=2020,yr2=2021,dx=3,dy=2,fact_dxy=2,station=stat) #ch4: coarser resolution

#those are going faster: 
read_cams.read_cams_eac4(dir_data_cams,station=stat)
read_cams.read_cams_egg4(dir_data_cams,yr1=2003,yr2=2020,station=stat)

## Read in the data

In [1]:
import numpy as np
import os,sys
import xarray as xr
import pandas as pd


In [4]:
import read_data
from read_data import AvailableData, create_data_reader

# Example file paths
data_path = "../data/"

# Instantiate different instrument readers
#wdc_reader = read_data.wdcGHGReader(data_path)
#ebas_reader = read_data.ebasReader(data_path)
# Create a list of BaseInstrumentReader objects
#instrument_readers = [wdc_reader, wdc_reader]

# print available data: 
# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
#print(all_data)
selected_data = ['CO2', 'CO2_flask', 'CO', 'CO_flask', 'CO_2002-2006', 'CH4', 'CH4_flask', 'O3'] # define data to read in. If empty, all data is used 

# TODO debug!

datasets = [] # list of all datasets to save as xarray-dataset

# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path,sel) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    data = data_reader.process_data(data)

    ds = data.to_xarray()
    ds = ds.assign_coords(species=sel)
    datasets.append(ds)

    #print(data)
    #print("\n")


Data from wdcGHGReader for CO2:
Data from wdcFlaskReader for CO2_flask:
Data from wdcGHGReader for CO:
Data from wdcFlaskReader for CO_flask:
Data from ebasReader for CO_2002-2006:


AttributeError: 'NoneType' object has no attribute 'rename_axis'

In [4]:
print(all_data)

['CO2', 'CO2_flask', 'CO', 'CO_flask', 'CO_2002-2006', 'CH4', 'CH4_flask', 'O3', 'aerosols_2015', 'aerosols', 'other_gases_ebas', 'other_gases_wdc', 'meteo']


In [1]:
create_data_reader(data_path,'CO2')

NameError: name 'create_data_reader' is not defined

In [2]:
import read_data
from read_data import AvailableData
data_path = "../data/"
spec='CO'
data_reader = read_data.create_data_reader(data_path,spec)
data = data_reader.read_data() 

In [4]:
data_reader.data_path

'../data/wdc/wdcgg/CO'

In [3]:
data

,site_gaw_id,year,month,day,hour,minute,second,year1,month1,day1,...,longitude,altitude,elevation,intake_height,flask_no,ORG_QCflag,QCflag,instrument,measurement_method,scale
time,,,,,,,,,,,,,,,,,,,,,
2002-05-31 23:00:00,MKN,2002,5,31,23,0,0,NaN,NaN,NaN,...,37.297199,3682.5,3678,4.5,NaN,NaN,3,1,9,93
2002-06-01 00:00:00,MKN,2002,6,1,0,0,0,NaN,NaN,NaN,...,37.297199,3682.5,3678,4.5,NaN,NaN,3,1,9,93
2002-06-01 01:00:00,MKN,2002,6,1,1,0,0,NaN,NaN,NaN,...,37.297199,3682.5,3678,4.5,NaN,NaN,2,1,9,93
2002-06-01 02:00:00,MKN,2002,6,1,2,0,0,NaN,NaN,NaN,...,37.297199,3682.5,3678,4.5,NaN,NaN,2,1,9,93
2002-06-01 03:00:00,MKN,2002,6,1,3,0,0,NaN,NaN,NaN,...,37.297199,3682.5,3678,4.5,NaN,NaN,2,1,9,93
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00,MKN,2022,12,31,19,0,0,NaN,NaN,NaN,...,37.297199,3688.0,3678,10.0,NaN,NaN,2,2,18,8
2022-12-31 20:00:00,MKN,2022,12,31,20,0,0,NaN,NaN,NaN,...,37.297199,3688.0,3678,10.0,NaN,NaN,2,2,18,8
2022-12-31 21:00:00,MKN,2022,12,31,21,0,0,NaN,NaN,NaN,...,37.297199,3688.0,3678,10.0,NaN,NaN,2,2,18,8
